# Step 2: View the stream

![Step 2 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/02-step.png)

## What you're building today

A connection to the camera. The phone by the feeder sends a continuous stream of pictures over WiFi. We open that stream and save one frame to disk.

By the end of this notebook you should see one bird (or whatever the phone is pointing at) saved as `data/snapshots/test.jpg`.

## Run in Colab

If you're reading this in Colab, you're already running. If not, click the badge above.

## Step 2.1 — What is a stream?

When you watch a video on your phone, the phone is showing you ~30 pictures per second. Each picture is a single still frame. Stack them fast enough and your eyes see motion.

A **stream** is just a way to get those frames one after another as fast as your network can carry them.

The old phone running IP Webcam broadcasts its frames on your local WiFi network. Anyone on the same WiFi can ask for them. We're going to ask.

In [ ]:
# Install the libraries we need
!pip install -q opencv-python-headless requests Pillow
print("OK")

## Step 2.2 — Where is the camera?

On your laptop: the phone is probably at something like `http://192.168.1.42:8080/video`. Find that IP in the IP Webcam app on the phone.

In Colab: the phone is unreachable (Colab runs in a Google data center, not your home WiFi). So we'll use a sample bird photo instead. The code is the same shape.

In [ ]:
# Two ways to grab a frame. Pick one.
import os

RUNNING_IN_COLAB = "COLAB_GPU" in os.environ

if RUNNING_IN_COLAB:
    # Colab path: use a sample bird photo (no phone needed)
    SOURCE = "https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/data/samples/sample-bird.jpg"
    print("Colab mode: using sample image")
else:
    # Local path: use the phone on your WiFi
    # Change this to match the IP shown in IP Webcam on your phone
    PHONE_IP = "192.168.1.42"  # <-- edit this!
    SOURCE = f"http://{PHONE_IP}:8080/photo.jpg"  # single frame endpoint
    print(f"Local mode: phone at {SOURCE}")

## Step 2.3 — Grab one frame

We ask the camera (or the sample image) for ONE picture, then we save it to disk.

In [ ]:
import requests
from pathlib import Path

# Make sure the snapshots folder exists
Path("data/snapshots").mkdir(parents=True, exist_ok=True)

# Grab one frame
response = requests.get(SOURCE, timeout=10)
response.raise_for_status()

# Save it
out_path = Path("data/snapshots/test.jpg")
out_path.write_bytes(response.content)

print(f"Saved {len(response.content)} bytes to {out_path}")
print(f"File exists: {out_path.exists()}")
print(f"Size on disk: {out_path.stat().st_size} bytes")

## Step 2.4 — Look at what we got

Numbers are fine, but we want to actually SEE the bird.

In [ ]:
from PIL import Image
from IPython.display import display

img = Image.open(out_path)
print(f"Size: {img.size}, mode: {img.mode}")
display(img)

## Done?

If you see an actual image (bird, scene, anything), the camera is wired up. 🎉

On your laptop: try changing the `PHONE_IP` and re-running Step 2.3 — does it grab a fresh frame from your phone?

## What's next

Issue **#3: Poll every N seconds**. Right now we grabbed ONE frame. Next we make a loop that grabs a new frame every few seconds, so the bird watcher is actually watching. 🐦